In [27]:
from pathlib import Path

import copernicusmarine
import pandas as pd
import xarray as xr


In [43]:
sites = pd.DataFrame(
    {
        "name": ("BARENTS", "HOT", "BATS", "PAPA", "GUAM"),
        "minimum_latitude": (74.12, 22.252, 31.104, 49.506, 12.501),
        "maximum_latitude": (75.120, 23.252, 32.104, 50.506, 13.501),
        "minimum_longitude": (26.469, -158.504, -64.700, -150.496, 149.495),
        "maximum_longitude": (27.469, -157.504, -63.700, -149.496, 150.495),
    }
)
date_start = pd.Timestamp.max = pd.Timestamp("1998-01-01")
date_end = pd.Timestamp.max = pd.Timestamp("2023-12-31")

sites["export_path"] = [f"./{station}_cmems.zarr" for station in sites["name"]]

In [44]:
physic_dataset_name = "cmems_mod_glo_bgc_my_0.083deg-lmtl-Fphy_PT1D-i"
bio_dataset_name = "cmems_mod_glo_bgc_my_0.083deg-lmtl_PT1D-i"


In [47]:
data = {}
for cpt, row in sites.iterrows():
    station = row["name"]
    export_path = Path(row["export_path"])
    export_path.mkdir(exist_ok=True, parents=True)

    physic = copernicusmarine.open_dataset(
        dataset_id=physic_dataset_name,
        minimum_latitude=row["minimum_latitude"],
        maximum_latitude=row["maximum_latitude"],
        minimum_longitude=row["minimum_longitude"],
        maximum_longitude=row["maximum_longitude"],
        start_datetime=date_start,
        end_datetime=date_end,
    )

    bio = copernicusmarine.open_dataset(
        dataset_id=bio_dataset_name,
        minimum_latitude=row["minimum_latitude"],
        maximum_latitude=row["maximum_latitude"],
        minimum_longitude=row["minimum_longitude"],
        maximum_longitude=row["maximum_longitude"],
        start_datetime=date_start,
        end_datetime=date_end,
    )
    data[station] = xr.merge([physic, bio])

INFO - 2025-05-28T12:57:46Z - Selected dataset version: "202411"
INFO - 2025-05-28T12:57:46Z - Selected dataset part: "default"
WARNING - 2025-05-28T12:57:48Z - Some of your subset selection [1997-12-31 23:00:00+00:00, 2023-12-30 23:00:00+00:00] for the time dimension exceed the dataset coordinates [1998-01-01 00:00:00+00:00, 2023-12-31 00:00:00+00:00]
INFO - 2025-05-28T12:57:51Z - Selected dataset version: "202411"
INFO - 2025-05-28T12:57:51Z - Selected dataset part: "default"
WARNING - 2025-05-28T12:57:53Z - Some of your subset selection [1997-12-31 23:00:00+00:00, 2023-12-30 23:00:00+00:00] for the time dimension exceed the dataset coordinates [1998-01-01 00:00:00+00:00, 2023-12-31 00:00:00+00:00]
INFO - 2025-05-28T12:57:57Z - Selected dataset version: "202411"
INFO - 2025-05-28T12:57:57Z - Selected dataset part: "default"
WARNING - 2025-05-28T12:57:59Z - Some of your subset selection [1997-12-31 23:00:00+00:00, 2023-12-30 23:00:00+00:00] for the time dimension exceed the dataset co

In [ ]:
for start, end in zip(date_start, date_end):
    if (not export_filepath.exists()) or (end not in xr.open_dataset(export_filepath, engine="zarr").time):
        coordinates.update({"start_datetime": start.strftime("%Y-%m-%d"), "end_datetime": end.strftime("%Y-%m-%d")})
        print(coordinates)

        physic = copernicusmarine.open_dataset(dataset_id=physic_dataset_name, **coordinates, variables=physic_variable)
        bio = copernicusmarine.open_dataset(dataset_id=bio_dataset_name, **coordinates, variables=bio_variable)

        physic = physic.interp(latitude=latitude, longitude=longitude, method="linear").load()
        bio = bio.interp(latitude=latitude, longitude=longitude, method="linear").load()

        results = xr.merge([physic, bio])

        if export_filepath.exists():
            results = xr.concat([xr.open_dataset(export_filepath, engine="zarr"), results], dim="time")
        results.to_zarr(export_filepath, mode="w")
